# HLA Typing Tool Accuracy Analysis
*Julia translation of `accuracy.ipynb`*

In [ ]:
using CSV
using DataFrames
using Statistics
using CairoMakie
using ColorSchemes

# Disable CairoMakie display in non-interactive mode (saves to file instead)
CairoMakie.activate!(type = "svg")

# Helper Functions
### Allele reformatting and scoring

In [ ]:
"""Trim allele to 1-field (`Gene*XX`) or 2-field (`Gene*XX:YY`) resolution."""
function reformat_allele(allele::AbstractString, resolution::Int=2)::String
    a = strip(allele)
    !occursin('*', a) && return a
    gene, rest = split(a, '*'; limit=2)
    parts = split(rest, ':')
    if resolution == 1
        return "$(gene)*$(parts[1])"
    else
        return length(parts) >= 2 ? "$(gene)*$(parts[1]):$(parts[2])" : "$(gene)*$(parts[1])"
    end
end

"""Return sorted locus names from a gold-standard DataFrame (all columns except the ID)."""
function get_dynamic_loci(gs_df::DataFrame)::Vector{String}
    sort(unique(first(split(c, '.')) for c in names(gs_df)[2:end]))
end

"""Build a per-locus set of all valid (reformatted) alleles found in the cohort GS."""
function build_global_valid(gs_df::DataFrame, resolution::Int=2)::Dict{String,Set{String}}
    loci = get_dynamic_loci(gs_df)
    valid = Dict(L => Set{String}() for L in loci)
    for row in eachrow(gs_df)
        for L in loci
            for suf in ("", ".1")
                col = L * suf
                col ∈ names(gs_df) || continue
                ismissing(row[col]) && continue
                for a in split(string(row[col]), '/')
                    push!(valid[L], reformat_allele(strip(a), resolution))
                end
            end
        end
    end
    return valid
end

"""
Score one allele pair: returns (correct, miscalled, novel, no_call).
- `gs_raw`  : gold-standard allele strings for this locus/sample
- `pr_raw`  : predicted allele strings (0, 1, or 2 values)
- `valid_L` : set of all cohort-valid alleles for this locus
"""
function score_pair(
        gs_raw::Vector{String},
        pr_raw::Vector{String},
        valid_L::Set{String},
        resolution::Int=2)::NTuple{4,Int}
    gs_set = Set(reformat_allele(a, resolution) for a in gs_raw)
    # normalise to exactly two prediction slots
    pair = if isempty(pr_raw)
        [nothing, nothing]
    elseif length(pr_raw) == 1
        [pr_raw[1], pr_raw[1]]
    else
        [pr_raw[1], pr_raw[2]]
    end
    correct = miscalled = novel = no_call = 0
    for allele in pair
        if isnothing(allele)
            no_call += 1
        else
            fmt = reformat_allele(allele, resolution)
            if fmt ∉ valid_L
                novel += 1
            elseif fmt ∈ gs_set
                correct += 1
            else
                miscalled += 1
            end
        end
    end
    return (correct, miscalled, novel, no_call)
end

"""Return the set of loci present in a prediction DataFrame (after _pre renaming)."""
function get_pr_loci(pr_df::DataFrame)::Set{String}
    s = Set{String}()
    for L in ["A","B","C","DRB1","DQB1"]
        (L*"_pre" ∈ names(pr_df) || L*".1_pre" ∈ names(pr_df)) && push!(s, L)
    end
    return s
end

### Core aggregation: per-tool, per-locus counts

In [ ]:
"""
Aggregate correct / miscalled / novel / no_call / total_calls counts
for every (tool, locus) combination across all requested datasets.

If `filter_option=false` (unfiltered), novel alleles are folded into miscalled.
If `filter_option=true`  (filtered),  novel alleles are kept separate.
"""
function aggregate_counts_per_tool_locus(
        tools::Vector{String},
        ds_nums,
        gs_dir::String,
        pr_dir::String;
        resolution::Int=2,
        filter_option::Bool=false)

    CLASSICAL = ["A","B","C","DRB1","DQB1"]
    blank = () -> Dict("correct"=>0,"miscalled"=>0,"novel"=>0,"no_call"=>0,"total_calls"=>0)
    counts = Dict(t => Dict(L => blank() for L in CLASSICAL) for t in tools)

    for t in tools, ds in ds_nums
        gs_path = joinpath(gs_dir, "$(ds)_gs.csv")
        pr_path = joinpath(pr_dir, "$(t)_d$(ds).csv")
        (isfile(gs_path) && isfile(pr_path)) || continue

        gs_df = CSV.read(gs_path, DataFrame; types=String)
        pr_df = CSV.read(pr_path, DataFrame; types=String)

        # strip ID columns
        gs_df[!, 1] = strip.(coalesce.(gs_df[!, 1], ""))
        pr_df[!, 1] = strip.(coalesce.(pr_df[!, 1], ""))

        # rename prediction columns to <name>_pre
        id_col = names(pr_df)[1]
        for c in names(pr_df)[2:end]
            rename!(pr_df, c => c*"_pre")
        end

        all_loci = get_dynamic_loci(gs_df)
        supported = get_pr_loci(pr_df)
        valid     = build_global_valid(gs_df, resolution)

        # inner join on sample ID
        merged = innerjoin(gs_df, pr_df;
                           on = names(gs_df)[1] => names(pr_df)[1],
                           makeunique=true)

        for row in eachrow(merged)
            # penalise unsupported loci as no-calls
            for L in CLASSICAL
                L ∈ all_loci && L ∉ supported || continue
                counts[t][L]["total_calls"] += 2
                counts[t][L]["no_call"]     += 2
            end

            # score supported loci
            for L in supported
                L ∈ all_loci || continue
                valid_L = get(valid, L, Set{String}())

                gs_raw = String[]
                for suf in ("", ".1")
                    col = L * suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        append!(gs_raw, strip.(split(string(row[col]), '/')))
                end
                isempty(gs_raw) && continue

                counts[t][L]["total_calls"] += 2

                pr_raw = String[]
                for suf in ("_pre", ".1_pre")
                    col = L * suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        push!(pr_raw, strip(string(row[col])))
                end

                c, m, n, nc = score_pair(gs_raw, pr_raw, valid_L, resolution)
                counts[t][L]["correct"]   += c
                counts[t][L]["miscalled"] += filter_option ? m : m + n
                counts[t][L]["novel"]     += filter_option ? n : 0
                counts[t][L]["no_call"]   += nc
            end
        end
    end
    return counts
end

In [ ]:
"""
Compute Class-I, Class-II and overall accuracy for each tool.
Returns a DataFrame with one row per tool.
"""
function compute_all_metrics(;
        tools::Vector{String},
        dataset_nums,
        gs_dir::String,
        pr_dir::String,
        resolution::Int=2,
        filter_option::Bool=false,
        with_raw_counts::Bool=false)::DataFrame

    CLASS_I  = ["A","B","C"]
    CLASS_II = ["DRB1","DQB1"]

    by_tool = aggregate_counts_per_tool_locus(
        tools, dataset_nums, gs_dir, pr_dir;
        resolution, filter_option)

    rows = []
    for t in tools
        lc = by_tool[t]
        ci      = sum(lc[L]["correct"]     for L in CLASS_I)
        calls_I = sum(lc[L]["total_calls"] for L in CLASS_I)
        cii     = sum(lc[L]["correct"]     for L in CLASS_II)
        calls_II = sum(lc[L]["total_calls"] for L in CLASS_II)
        calls_all = calls_I + calls_II

        row = (
            Tool        = t,
            ClassI_acc  = calls_I  > 0 ? ci  / calls_I  : missing,
            ClassII_acc = calls_II > 0 ? cii / calls_II : missing,
            Overall_acc = calls_all > 0 ? (ci + cii) / calls_all : missing,
        )
        if with_raw_counts
            row = merge(row, (
                ClassI_Correct    = ci,
                ClassI_Miscalled  = sum(lc[L]["miscalled"] for L in CLASS_I),
                ClassI_Novel      = sum(lc[L]["novel"]     for L in CLASS_I),
                ClassI_NoCall     = sum(lc[L]["no_call"]   for L in CLASS_I),
                ClassI_TotalCalls = calls_I,
                ClassII_Correct   = cii,
                ClassII_Miscalled = sum(lc[L]["miscalled"] for L in CLASS_II),
                ClassII_Novel     = sum(lc[L]["novel"]     for L in CLASS_II),
                ClassII_NoCall    = sum(lc[L]["no_call"]   for L in CLASS_II),
                ClassII_TotalCalls= calls_II,
                TotalCorrect      = ci + cii,
                TotalMiscalled    = sum(lc[L]["miscalled"] for L in ["A","B","C","DRB1","DQB1"]),
                TotalNovel        = sum(lc[L]["novel"]     for L in ["A","B","C","DRB1","DQB1"]),
                TotalNoCall       = sum(lc[L]["no_call"]   for L in ["A","B","C","DRB1","DQB1"]),
                TotalCalls        = calls_all,
            ))
        end
        push!(rows, row)
    end
    return DataFrame(rows)
end

In [ ]:
"""Per-locus accuracy breakdown for each tool — used for Figure 5."""
function compute_locus_metrics(;
        tools::Vector{String},
        dataset_nums,
        gs_dir::String,
        pr_dir::String,
        resolution::Int=2,
        filter_option::Bool=false)::DataFrame

    by_tool = aggregate_counts_per_tool_locus(
        tools, dataset_nums, gs_dir, pr_dir;
        resolution, filter_option)

    rows = []
    for t in tools, (L, cts) in by_tool[t]
        total = cts["total_calls"]
        push!(rows, (
            Tool            = t,
            Locus           = L,
            CorrectAlleles  = cts["correct"],
            MiscalledAlleles= cts["miscalled"],
            NovelAlleles    = cts["novel"],
            NoCallAlleles   = cts["no_call"],
            TotalAlleles    = total,
            Accuracy        = total > 0 ? cts["correct"] / total : missing,
        ))
    end
    return sort!(DataFrame(rows), [:Tool, :Locus])
end

# Color Palette

In [ ]:
const PALETTE = Dict{String,String}(
    # Resolution
    "1st field"         => "#A1D99B",
    "2nd field"         => "#31A354",
    # Call categories
    "Correct"           => "#4CAF50",
    "Miscalled"         => "#FF9800",
    "Novel"             => "#9C27B0",
    "No-call"           => "#F44336",
    # Loci
    "A"                 => "#e41a1c",
    "B"                 => "#377eb8",
    "C"                 => "#4daf4a",
    "DQB1"              => "#984ea3",
    "DRB1"              => "#ff7f00",
    # Populations
    "Europe"            => "#1b9e77",
    "Yoruba"            => "#d95f02",
    # Tools
    "arcas"             => "#80b1d3",
    "optitype"          => "#8dd3c7",
    "seq2hla"           => "#fccde5",
    "hlaminer"          => "#fdb462",
    "phlat"             => "#bebada",
    "hlavbseq"          => "#d9d9d9",
    "hlahd"             => "#ffffb3",
    "hisat"             => "#bc80bd",
    "hlaforest"         => "#fb8072",
    "hlapers"           => "#ccebc5",
    "T1K"               => "#ffed6f",
    "rna2hla"           => "#b3de69",
)

tool_color(t) = get(PALETTE, t, "#888888")

# Configuration

In [ ]:
const TOOLS    = ["arcas","optitype","seq2hla","hlaminer",
                  "phlat","hlavbseq","hlahd","hisat","hlaforest","hlapers","T1K"]
const DS_NUMS  = 1:6
const GS_DIR   = "../datasets"
const PR_DIR   = "../results/standard"

# Class I-only tools: excluded from Class II no-call penalty
const CLASS_I_ONLY_TOOLS = Set(["optitype"])
println("Configuration loaded: $(length(TOOLS)) tools, datasets $(first(DS_NUMS))–$(last(DS_NUMS))")

# Accuracy by Dataset

In [ ]:
function calculate_dataset_accuracy(;
        tools=TOOLS, ds_nums=DS_NUMS,
        gs_dir=GS_DIR, pr_dir=PR_DIR, resolution=2)

    rows = []
    for t in tools, ds in ds_nums
        gs_path = joinpath(gs_dir, "$(ds)_gs.csv")
        pr_path = joinpath(pr_dir, "$(t)_d$(ds).csv")
        (isfile(gs_path) && isfile(pr_path)) || continue

        gs_df = CSV.read(gs_path, DataFrame; types=String)
        pr_df = CSV.read(pr_path, DataFrame; types=String)

        gs_df[!, 1] = strip.(coalesce.(gs_df[!, 1], ""))
        pr_df[!, 1] = strip.(coalesce.(pr_df[!, 1], ""))
        for c in names(pr_df)[2:end]
            rename!(pr_df, c => c*"_pre")
        end

        all_loci  = get_dynamic_loci(gs_df)
        supported = get_pr_loci(pr_df)
        valid     = build_global_valid(gs_df, resolution)
        merged    = innerjoin(gs_df, pr_df;
                              on=names(gs_df)[1]=>names(pr_df)[1], makeunique=true)

        correct = no_calls = total = 0
        for row in eachrow(merged)
            for L in all_loci
                valid_L = get(valid, L, Set{String}())
                gs_raw = String[]
                for suf in ("", ".1")
                    col = L * suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        append!(gs_raw, strip.(split(string(row[col]), '/')))
                end
                isempty(gs_raw) && continue
                total += 2
                if L ∉ supported
                    no_calls += 2
                    continue
                end
                pr_raw = String[]
                for suf in ("_pre", ".1_pre")
                    col = L * suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        push!(pr_raw, strip(string(row[col])))
                end
                c, m, n, nc = score_pair(gs_raw, pr_raw, valid_L, resolution)
                correct  += c
                no_calls += nc
            end
        end
        push!(rows, (Tool=t, Dataset=ds,
                     Accuracy   = total > 0 ? correct / total : missing,
                     Correct    = correct,
                     Total      = total,
                     NoCalls    = no_calls))
    end
    return DataFrame(rows)
end

df_by_dataset = calculate_dataset_accuracy()
println(df_by_dataset)

# Figure 3 & 4: Bulk Accuracy Metrics

In [ ]:
df_unf_1 = compute_all_metrics(tools=TOOLS, dataset_nums=DS_NUMS,
                                gs_dir=GS_DIR, pr_dir=PR_DIR,
                                resolution=1, filter_option=false, with_raw_counts=true)
df_flt_1 = compute_all_metrics(tools=TOOLS, dataset_nums=DS_NUMS,
                                gs_dir=GS_DIR, pr_dir=PR_DIR,
                                resolution=1, filter_option=true,  with_raw_counts=true)
df_unf_2 = compute_all_metrics(tools=TOOLS, dataset_nums=DS_NUMS,
                                gs_dir=GS_DIR, pr_dir=PR_DIR,
                                resolution=2, filter_option=false, with_raw_counts=true)
df_flt_2 = compute_all_metrics(tools=TOOLS, dataset_nums=DS_NUMS,
                                gs_dir=GS_DIR, pr_dir=PR_DIR,
                                resolution=2, filter_option=true,  with_raw_counts=true)

# Assemble long-format df_plot
function prep(df, mode, field)
    d = copy(df)
    d[!, :Mode]       .= mode
    d[!, :Field]      .= field
    d[!, :Accuracy]    = d.Overall_acc
    d[!, :NoCallRate]  = d.TotalNoCall ./ d.TotalCalls
    return d
end

df_plot = vcat(
    prep(df_unf_1, "pre-filter", "1-field"),
    prep(df_flt_1, "filtered",   "1-field"),
    prep(df_unf_2, "pre-filter", "2-field"),
    prep(df_flt_2, "filtered",   "2-field"),
)

# Tool order: ascending 2-field unfiltered accuracy
tools_order = Vector{String}(sort(
    filter(:Mode => ==("pre-filter"), filter(:Field => ==("2-field"), df_plot)),
    :Accuracy
).Tool)

println("Metrics computed. Tool order (2-field asc): ", tools_order)

## Figure 3a: Bulk Accuracy (Unfiltered)

In [ ]:
let
    # pivot unfiltered 1-field and 2-field
    unf = filter(:Mode => ==("pre-filter"), df_plot)
    acc1 = Dict(r.Tool => coalesce(r.Accuracy, 0.0)
                for r in eachrow(filter(:Field => ==("1-field"), unf)))
    acc2 = Dict(r.Tool => coalesce(r.Accuracy, 0.0)
                for r in eachrow(filter(:Field => ==("2-field"), unf)))

    n = length(tools_order)
    xs = 1:n
    w  = 0.35

    fig = Figure(size=(900, 600))
    ax  = Axis(fig[1,1];
               xticks = (xs, tools_order),
               xticklabelrotation = π/4,
               ylabel = "Accuracy",
               title  = "Figure 3a: Bulk Accuracy (Unfiltered)")

    barplot!(ax, xs .- w/2, [acc1[t] for t in tools_order];
             width=w, color=PALETTE["1st field"], label="1-field")
    barplot!(ax, xs .+ w/2, [acc2[t] for t in tools_order];
             width=w, color=PALETTE["2nd field"], label="2-field")

    axislegend(ax; position=:lb)
    ylims!(ax, 0, 1)
    save("figures_julia/fig3a_bulk_accuracy.svg", fig)
    fig
end

## Figure 3b: Class I ↔ Class II Accuracy (Filtered)

In [ ]:
let
    flt = filter(:Mode => ==("filtered"), df_plot)
    acc_I1  = Dict(r.Tool => coalesce(r.ClassI_acc,  0.0) for r in eachrow(filter(:Field => ==("1-field"), flt)))
    acc_I2  = Dict(r.Tool => coalesce(r.ClassI_acc,  0.0) for r in eachrow(filter(:Field => ==("2-field"), flt)))
    acc_II1 = Dict(r.Tool => coalesce(r.ClassII_acc, 0.0) for r in eachrow(filter(:Field => ==("1-field"), flt)))
    acc_II2 = Dict(r.Tool => coalesce(r.ClassII_acc, 0.0) for r in eachrow(filter(:Field => ==("2-field"), flt)))

    n  = length(tools_order)
    ys = 1:n
    w  = 0.3

    fig = Figure(size=(1000, 600))
    axL = Axis(fig[1,1]; title="Class I",  ylabel="",
               yticks=(ys, tools_order), yticklabelsize=12)
    axR = Axis(fig[1,2]; title="Class II", ylabel="",
               yticklabelsvisible=false, yticklabelsize=12)

    # Class I — bars go LEFT (negative)
    barplot!(axL, ys .- w/2, [-acc_I1[t] for t in tools_order];
             direction=:x, width=w, color=PALETTE["1st field"], label="1-field")
    barplot!(axL, ys .+ w/2, [-acc_I2[t] for t in tools_order];
             direction=:x, width=w, color=PALETTE["2nd field"], label="2-field")
    xlims!(axL, -1, 0)
    axL.xtickformat = tv -> string.(round.(abs.(tv); digits=2))

    # Class II — bars go RIGHT
    barplot!(axR, ys .- w/2, [acc_II1[t] for t in tools_order];
             direction=:x, width=w, color=PALETTE["1st field"], label="1-field")
    barplot!(axR, ys .+ w/2, [acc_II2[t] for t in tools_order];
             direction=:x, width=w, color=PALETTE["2nd field"], label="2-field")
    xlims!(axR, 0, 1)

    Legend(fig[2, :], axL; orientation=:horizontal, tellwidth=false)
    save("figures_julia/fig3b_classI_classII.svg", fig)
    fig
end

## Figure 4: Fraction of Calls (Unfiltered vs Filtered)

In [ ]:
let
    fu = Dict(r.Tool => r for r in eachrow(df_unf_2))
    ff = Dict(r.Tool => r for r in eachrow(df_flt_2))

    n   = length(tools_order)
    xs  = 1:n

    fig = Figure(size=(1200, 500))
    ax1 = Axis(fig[1,1]; title="Unfiltered",
               xticks=(xs, tools_order), xticklabelrotation=π/4,
               ylabel="Fraction of calls")
    ax2 = Axis(fig[1,2]; title="Filtered",
               xticks=(xs, tools_order), xticklabelrotation=π/4)

    for (ax, src, cats) in [
        (ax1, fu, [("Correct","TotalCorrect"),("Miscalled","TotalMiscalled"),("No-call","TotalNoCall")]),
        (ax2, ff, [("Correct","TotalCorrect"),("Miscalled","TotalMiscalled"),("Novel","TotalNovel"),("No-call","TotalNoCall")]),
    ]
        bottom = zeros(n)
        for (label, col) in cats
            vals = [coalesce(src[t][Symbol(col)], 0) / max(coalesce(src[t][:TotalCalls], 1), 1)
                    for t in tools_order]
            barplot!(ax, xs, vals; offset=copy(bottom), color=PALETTE[label], label=label)
            bottom .+= vals
        end
        ylims!(ax, 0, 1)
    end

    leg_patches = [PolyElement(color=PALETTE[l], strokecolor=:transparent)
                   for l in ["Correct","Miscalled","Novel","No-call"]]
    Legend(fig[2, :], leg_patches, ["Correct","Miscalled","Novel","No-call"];
           orientation=:horizontal, tellwidth=false)
    save("figures_julia/fig4_fraction_calls.svg", fig)
    fig
end

# Figure 14: No-Call Rates and Novel Alleles

In [ ]:
# Get per-locus data (needed for per-locus no-call heatmap)
by_tool_locus = aggregate_counts_per_tool_locus(
    TOOLS, DS_NUMS, GS_DIR, PR_DIR; resolution=2, filter_option=false)

LOCI        = ["A","B","C","DRB1","DQB1"]
CLASS_I_L   = ["A","B","C"]
CLASS_II_L  = ["DRB1","DQB1"]

# No-call rate per tool (Class II excluded for Class I-only tools)
no_call_rows = []
for t in TOOLS
    total_nc = total_c = 0
    for L in LOCI
        (t ∈ CLASS_I_ONLY_TOOLS && L ∈ CLASS_II_L) && continue
        total_c  += by_tool_locus[t][L]["total_calls"]
        total_nc += by_tool_locus[t][L]["no_call"]
    end
    push!(no_call_rows, (Tool=t,
                         TotalNoCall=total_nc,
                         TotalCalls =total_c,
                         NoCallRate = total_c > 0 ? total_nc / total_c * 100 : 0.0))
end
no_call_summary = sort!(DataFrame(no_call_rows), :NoCallRate)

# Per-locus heatmap matrix
heatmap_mat = [
    let d = by_tool_locus[t][L]
        d["total_calls"] > 0 ? d["no_call"] / d["total_calls"] * 100 : 0.0
    end
    for t in no_call_summary.Tool, L in LOCI
]  # rows=tools (sorted), cols=loci

# Novel allele summary
novel_rows = [(Tool=r.Tool,
               NovelRate = r.TotalCalls > 0 ? r.TotalNovel / r.TotalCalls * 100 : 0.0,
               TotalNovel=r.TotalNovel, TotalCalls=r.TotalCalls)
              for r in eachrow(df_flt_2)]
novel_summary = sort!(DataFrame(novel_rows), :NovelRate)

# Filtered vs unfiltered accuracy comparison
comparison = innerjoin(
    select(df_unf_2, :Tool, :Overall_acc => :Unfiltered_Acc),
    select(df_flt_2, :Tool, :Overall_acc => :Filtered_Acc),
    on=:Tool)
comparison[!, :Impact] = (coalesce.(comparison.Filtered_Acc, 0.0) .-
                           coalesce.(comparison.Unfiltered_Acc, 0.0)) .* 100
sort!(comparison, :Impact; rev=true)

println("Figure 14 data ready.")
println("No-call range: $(round(minimum(no_call_summary.NoCallRate), digits=1))% – ",
        "$(round(maximum(no_call_summary.NoCallRate), digits=1))%")

In [ ]:
let
    sorted_tools = no_call_summary.Tool
    n = length(sorted_tools)

    fig = Figure(size=(900, 1400))

    # ── Panel A: No-call rate bar ──────────────────────────────────────────
    ax1 = Axis(fig[1,1]; title="Fig 14a: No-Call Rate by Tool",
               xlabel="No-Call Rate (%)",
               yticks=(1:n, sorted_tools))
    barplot!(ax1, 1:n, no_call_summary.NoCallRate;
             direction=:x, color=PALETTE["No-call"])
    xlims!(ax1, 0, 100)

    # ── Panel B: No-call heatmap ────────────────────────────────────────────
    ax2 = Axis(fig[2,1]; title="Fig 14b: No-Call Rate by Tool and Locus",
               xticks=(1:5, LOCI),
               yticks=(1:n, sorted_tools))
    hm = heatmap!(ax2, 1:5, 1:n, heatmap_mat';
                  colormap=:Reds, colorrange=(0, 100))
    Colorbar(fig[2,2], hm; label="No-Call Rate (%)")
    for (i, L) in enumerate(LOCI), (j, t) in enumerate(sorted_tools)
        text!(ax2, i, j; text=string(round(heatmap_mat[j,i]; digits=1)),
              align=(:center,:center), fontsize=10)
    end

    # ── Panel C: Novel allele rate ──────────────────────────────────────────
    ax3 = Axis(fig[3,1]; title="Fig 14c: Novel Allele Rate by Tool",
               xlabel="Novel Allele Rate (%)",
               yticks=(1:n, novel_summary.Tool))
    barplot!(ax3, 1:n, novel_summary.NovelRate;
             direction=:x, color=PALETTE["Novel"])

    # ── Panel D: Unfiltered vs Filtered accuracy ────────────────────────────
    m   = nrow(comparison)
    xs  = 1:m
    w   = 0.3
    ax4 = Axis(fig[4,1]; title="Fig 14d: Impact of Novel Alleles on Accuracy",
               xticks=(xs, comparison.Tool), xticklabelrotation=π/4,
               ylabel="Accuracy (%)")
    barplot!(ax4, xs .- w/2, coalesce.(comparison.Unfiltered_Acc, 0.0) .* 100;
             width=w, color=PALETTE["Miscalled"], label="Unfiltered")
    barplot!(ax4, xs .+ w/2, coalesce.(comparison.Filtered_Acc,   0.0) .* 100;
             width=w, color=PALETTE["Correct"],   label="Filtered")
    axislegend(ax4; position=:lb)
    ylims!(ax4, 0, 100)

    # ── Panel E: Accuracy vs Novel Rate scatter ─────────────────────────────
    tradeoff = innerjoin(
        select(df_flt_2, :Tool, :Overall_acc),
        select(DataFrame(novel_summary), :Tool, :NovelRate),
        on=:Tool)
    ax5 = Axis(fig[5,1]; title="Fig 14e: Accuracy vs Novel Rate",
               xlabel="Novel Allele Rate (%)", ylabel="Filtered Accuracy (%)")
    for r in eachrow(tradeoff)
        scatter!(ax5, r.NovelRate, coalesce(r.Overall_acc, 0.0)*100;
                 markersize=14, color=tool_color(r.Tool), strokecolor=:black, strokewidth=1)
        text!(ax5, r.NovelRate + 0.5, coalesce(r.Overall_acc, 0.0)*100;
              text=r.Tool, fontsize=10)
    end
    ylims!(ax5, 0, 100)

    save("figures_julia/fig14_nocall_novel.svg", fig)
    fig
end

# Figure 5: Accuracy per Locus per Tool

In [ ]:
df_locus = compute_locus_metrics(
    tools=TOOLS, dataset_nums=DS_NUMS,
    gs_dir=GS_DIR, pr_dir=PR_DIR,
    resolution=2, filter_option=true)

println(df_locus)

In [ ]:
let
    LOCI_ORDER = ["A","B","C","DRB1","DQB1"]

    # Build accuracy matrix (tools × loci)
    acc_dict = Dict((r.Tool, r.Locus) => coalesce(r.Accuracy, 0.0)
                    for r in eachrow(df_locus))
    # Sort tools by locus A accuracy descending
    sorted_tools = sort(TOOLS;
        by = t -> -get(acc_dict, (t,"A"), 0.0))

    nt, nl = length(sorted_tools), length(LOCI_ORDER)
    mat = [get(acc_dict, (t, L), 0.0) for t in sorted_tools, L in LOCI_ORDER]

    fig = Figure(size=(700, 600))
    ax  = Axis(fig[1,1];
               title  = "Two-field HLA Accuracy by Tool and Locus",
               xticks = (1:nl, LOCI_ORDER),
               yticks = (1:nt, sorted_tools),
               yticklabelsize = 12)
    hm  = heatmap!(ax, 1:nl, 1:nt, mat';
                   colormap=:YlGnBu, colorrange=(0,1))
    Colorbar(fig[1,2], hm; label="Accuracy")
    for (i, L) in enumerate(LOCI_ORDER), (j, t) in enumerate(sorted_tools)
        text!(ax, i, j; text=string(round(mat[j,i]; digits=2)),
              align=(:center,:center), fontsize=9,
              color = mat[j,i] > 0.5 ? :black : :white)
    end
    save("figures_julia/fig5_locus_heatmap.svg", fig)
    fig
end

# Figure 6: Miscalled Alleles per Locus per Tool

In [ ]:
# Fig 6a — miscalled allele COUNTS (grouped horizontal bar)
let
    LOCI_ORDER = ["A","B","C","DRB1","DQB1"]

    # order tools by total miscalls descending
    misc_tot = Dict(t => sum(
        coalesce(filter(r -> r.Tool==t, df_locus).MiscalledAlleles, 0))
        for t in TOOLS)
    tool_order6 = sort(TOOLS; by=t -> -misc_tot[t])

    n, nl = length(tool_order6), length(LOCI_ORDER)
    w  = 0.15

    fig = Figure(size=(900, 600))
    ax  = Axis(fig[1,1];
               title  = "Fig 6a: Miscalled Alleles per Locus by Tool",
               xlabel = "Miscalled Alleles (summed across datasets)",
               yticks = (1:n, tool_order6))

    for (il, L) in enumerate(LOCI_ORDER)
        vals = Float64[]
        for t in tool_order6
            sub = filter(r -> r.Tool==t && r.Locus==L, df_locus)
            push!(vals, isempty(sub) ? 0.0 : Float64(sub[1, :MiscalledAlleles]))
        end
        offset = (il - 3) * w
        barplot!(ax, (1:n) .+ offset, vals;
                 direction=:x, width=w*0.9, color=PALETTE[L], label=L)
    end
    axislegend(ax; position=:rb)
    save("figures_julia/fig6a_miscalled_count.svg", fig)
    fig
end

In [ ]:
# Fig 6b — miscalled allele PROPORTIONS (stacked horizontal bar)
let
    LOCI_ORDER = ["A","B","C","DRB1","DQB1"]

    misc_tot = Dict(t => sum(coalesce(filter(r -> r.Tool==t, df_locus).MiscalledAlleles, 0))
                    for t in TOOLS)
    tool_order6 = sort(TOOLS; by=t -> -misc_tot[t])
    n = length(tool_order6)

    fig = Figure(size=(900, 600))
    ax  = Axis(fig[1,1];
               title  = "Fig 6b: Proportion of Miscalled Alleles per Locus",
               xlabel = "Proportion of total miscalls",
               yticks = (1:n, tool_order6))

    bottom = zeros(n)
    for L in LOCI_ORDER
        tot  = [max(misc_tot[t], 1) for t in tool_order6]
        vals = Float64[]
        for t in tool_order6
            sub = filter(r -> r.Tool==t && r.Locus==L, df_locus)
            push!(vals, isempty(sub) ? 0.0 : Float64(sub[1,:MiscalledAlleles]) / max(misc_tot[t],1))
        end
        barplot!(ax, 1:n, vals; direction=:x, offset=copy(bottom),
                 color=PALETTE[L], label=L)
        bottom .+= vals
    end
    axislegend(ax; position=:rb)
    save("figures_julia/fig6b_miscalled_proportion.svg", fig)
    fig
end

# Figure 7: Misclassification Rate by Tool

In [ ]:
# Fig 7 — overall misclassification rate sorted bar
let
    mis_rate = Dict(t =>
        let sub = filter(r -> r.Tool==t, df_locus)
            tot  = sum(sub.TotalAlleles)
            mis  = sum(sub.MiscalledAlleles)
            tot > 0 ? mis / tot : 0.0
        end for t in TOOLS)

    sorted = sort(TOOLS; by=t -> mis_rate[t], rev=true)
    n = length(sorted)

    fig = Figure(size=(800, 500))
    ax  = Axis(fig[1,1];
               title  = "Fig 7: Misclassification Rate by Tool (2-field filtered)",
               ylabel = "Misclassification Rate",
               xticks = (1:n, sorted),
               xticklabelrotation = π/4)
    barplot!(ax, 1:n, [mis_rate[t] for t in sorted];
             color = [tool_color(t) for t in sorted])
    ylims!(ax, 0, nothing)
    save("figures_julia/fig7_misclassification_rate.svg", fig)
    fig
end

# Figure 8: Accuracy by Read Length

In [ ]:
function compute_readlength_accuracy(;
        tools::Vector{String}=TOOLS,
        read_lengths=[36, 51, 76, 101, 126],
        gs_file::String  = "../datasets/readlength_gs.csv",
        results_dir::String = "../results/read_length/standard",
        resolution::Int=2)::DataFrame

    isfile(gs_file) || (println("Missing: $gs_file"); return DataFrame())

    gs_df  = CSV.read(gs_file, DataFrame; types=String)
    gs_df[!, 1] = strip.(coalesce.(gs_df[!, 1], ""))
    id_col = names(gs_df)[1]

    all_loci = get_dynamic_loci(gs_df)
    valid    = build_global_valid(gs_df, resolution)

    rows = []
    for t in tools, rl in read_lengths
        pr_path = joinpath(results_dir, "$(t)_$(rl).csv")
        isfile(pr_path) || continue

        pr_df = CSV.read(pr_path, DataFrame; types=String)
        pr_df[!, 1] = strip.(coalesce.(pr_df[!, 1], ""))
        for c in names(pr_df)[2:end]
            rename!(pr_df, c => c*"_pre")
        end

        merged = innerjoin(gs_df, pr_df;
                           on=names(gs_df)[1]=>names(pr_df)[1], makeunique=true)

        correct = total = 0
        for row in eachrow(merged)
            for L in all_loci
                gs_raw = String[]
                for suf in ("",".1")
                    col = L*suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        append!(gs_raw, strip.(split(string(row[col]),'/')))
                end
                isempty(gs_raw) && continue
                total += 2
                pr_raw = String[]
                for suf in ("_pre",".1_pre")
                    col = L*suf
                    col ∈ names(merged) && !ismissing(row[col]) &&
                        push!(pr_raw, strip(string(row[col])))
                end
                c, m, n, nc = score_pair(gs_raw, pr_raw, get(valid,L,Set{String}()), resolution)
                correct += c
            end
        end
        push!(rows, (Tool=t, ReadLength=rl,
                     Accuracy = total > 0 ? correct/total : missing))
    end
    return DataFrame(rows)
end

df_rl = compute_readlength_accuracy()
println(df_rl)

In [ ]:
# Fig 8 — line plot: accuracy vs read length
let
    isempty(df_rl) && (println("No read-length data"); return)

    rls      = sort(unique(df_rl.ReadLength))
    tools_rl = unique(df_rl.Tool)

    fig = Figure(size=(800, 500))
    ax  = Axis(fig[1,1];
               title  = "Fig 8: Accuracy by Read Length",
               xlabel = "Read Length (bp)",
               ylabel = "Accuracy",
               xticks = (1:length(rls), string.(rls)))

    for t in tools_rl
        sub  = filter(r -> r.Tool==t, df_rl)
        sort!(sub, :ReadLength)
        xs   = [findfirst(==(rl), rls) for rl in sub.ReadLength]
        ys   = coalesce.(sub.Accuracy, NaN)
        lines!(ax, xs, ys; color=tool_color(t), linewidth=2, label=t)
        scatter!(ax, xs, ys; color=tool_color(t), markersize=8)
    end

    axislegend(ax; position=:lb, nbanks=2)
    ylims!(ax, 0, 1)
    save("figures_julia/fig8_readlength_accuracy.svg", fig)
    fig
end

# Figure 9: Ancestry Accuracy (Europe vs Yoruba)

In [ ]:
# Build European and Yoruban GS subsets from dataset 1
function build_ancestry_gs(;
        sra_table  = "../datasets/archive/SraRunTableD1.txt",
        gs_file    = "../datasets/1_gs.csv")

    (isfile(sra_table) && isfile(gs_file)) || return nothing, nothing

    sra = CSV.read(sra_table, DataFrame)
    gs  = CSV.read(gs_file,   DataFrame; types=String)

    id_col = names(gs)[1]   # "Run"
    eur_pops = Set(["CEU","FIN","GBR","TSI"])

    sra_run = string.(sra.Run)
    eur_runs = Set(sra_run[in.(string.(sra.Population), Ref(eur_pops))])
    yri_runs = Set(sra_run[string.(sra.Population) .== "YRI"])

    gs_eur = filter(r -> string(r[id_col]) ∈ eur_runs, gs)
    gs_yri = filter(r -> string(r[id_col]) ∈ yri_runs, gs)

    println("EUR samples: $(nrow(gs_eur)), YRI samples: $(nrow(gs_yri))")
    return gs_eur, gs_yri
end

function compute_ancestry_accuracy(gs_sub::DataFrame, tool::String;
        pr_dir=PR_DIR, resolution::Int=2)

    pr_path = joinpath(pr_dir, "$(tool)_d1.csv")
    isfile(pr_path) || return missing, missing

    pr_df = CSV.read(pr_path, DataFrame; types=String)
    pr_df[!, 1] = strip.(coalesce.(pr_df[!, 1], ""))
    for c in names(pr_df)[2:end]
        rename!(pr_df, c => c*"_pre")
    end

    gs_sub2 = copy(gs_sub)
    gs_sub2[!, 1] = strip.(coalesce.(gs_sub2[!, 1], ""))

    all_loci = get_dynamic_loci(gs_sub2)
    valid    = build_global_valid(gs_sub2, resolution)

    merged = innerjoin(gs_sub2, pr_df;
                       on=names(gs_sub2)[1]=>names(pr_df)[1], makeunique=true)
    isempty(merged) && return missing, missing

    ci = ti = cii = tii = 0
    CLASS_I_SET  = Set(["A","B","C"])
    CLASS_II_SET = Set(["DRB1","DQB1"])

    for row in eachrow(merged)
        for L in all_loci
            gs_raw = String[]
            for suf in ("",".1")
                col = L*suf
                col ∈ names(merged) && !ismissing(row[col]) &&
                    append!(gs_raw, strip.(split(string(row[col]),'/')))
            end
            isempty(gs_raw) && continue
            pr_raw = String[]
            for suf in ("_pre",".1_pre")
                col = L*suf
                col ∈ names(merged) && !ismissing(row[col]) &&
                    push!(pr_raw, strip(string(row[col])))
            end
            c, m, n, nc = score_pair(gs_raw, pr_raw, get(valid,L,Set{String}()), resolution)
            if L ∈ CLASS_I_SET;  ci += c; ti += 2; end
            if L ∈ CLASS_II_SET; cii += c; tii += 2; end
        end
    end
    acc_I  = ti  > 0 ? ci  / ti  : missing
    acc_II = tii > 0 ? cii / tii : missing
    return acc_I, acc_II
end

gs_eur, gs_yri = build_ancestry_gs()

anc_rows = []
if !isnothing(gs_eur)
    for t in TOOLS, res in [1, 2]
        ai_e, aii_e = compute_ancestry_accuracy(gs_eur, t; resolution=res)
        ai_y, aii_y = compute_ancestry_accuracy(gs_yri, t; resolution=res)
        push!(anc_rows, (Tool=t, Resolution=res,
                         ClassI_Eur=coalesce(ai_e,0.0), ClassI_Yri=coalesce(ai_y,0.0),
                         ClassII_Eur=coalesce(aii_e,0.0), ClassII_Yri=coalesce(aii_y,0.0)))
    end
end
df_anc = DataFrame(anc_rows)
println(df_anc)

In [ ]:
# Fig 9 — 4-panel bar chart: 1/2-field × Class I/II, EUR vs YRI
let
    isempty(df_anc) && (println("No ancestry data"); return)

    fig   = Figure(size=(1200, 700))
    n     = length(TOOLS)
    xs    = 1:n
    w     = 0.35

    configs = [
        (1, "1-field Class I",  :ClassI_Eur,  :ClassI_Yri,  1),
        (2, "1-field Class II", :ClassII_Eur, :ClassII_Yri, 1),
        (3, "2-field Class I",  :ClassI_Eur,  :ClassI_Yri,  2),
        (4, "2-field Class II", :ClassII_Eur, :ClassII_Yri, 2),
    ]

    for (panel, title, col_e, col_y, res) in configs
        row_idx, col_idx = divrem(panel-1, 2) .+ (1, 1)
        ax = Axis(fig[row_idx, col_idx]; title=title,
                  xticks=(xs, TOOLS), xticklabelrotation=π/4,
                  ylabel="Accuracy")
        sub = filter(r -> r.Resolution==res, df_anc)
        isempty(sub) && continue
        d = Dict(r.Tool => r for r in eachrow(sub))
        eur_v = [haskey(d,t) ? coalesce(d[t][col_e], 0.0) : 0.0 for t in TOOLS]
        yri_v = [haskey(d,t) ? coalesce(d[t][col_y], 0.0) : 0.0 for t in TOOLS]
        barplot!(ax, xs .- w/2, eur_v; width=w, color=PALETTE["Europe"], label="European")
        barplot!(ax, xs .+ w/2, yri_v; width=w, color=PALETTE["Yoruba"], label="Yoruba")
        ylims!(ax, 0, 1.05)
    end

    leg = [PolyElement(color=PALETTE[k], strokecolor=:transparent) for k in ["Europe","Yoruba"]]
    Legend(fig[3,:], leg, ["European","Yoruba"]; orientation=:horizontal, tellwidth=false)
    save("figures_julia/fig9_ancestry_accuracy.svg", fig)
    fig
end

# Figure 10: Ancestry Accuracy by Locus

In [ ]:
function compute_ancestry_locus_accuracy(gs_sub::DataFrame, tool::String;
        pr_dir=PR_DIR, resolution::Int=2)::Dict{String,Float64}

    pr_path = joinpath(pr_dir, "$(tool)_d1.csv")
    isfile(pr_path) || return Dict{String,Float64}()

    pr_df = CSV.read(pr_path, DataFrame; types=String)
    pr_df[!, 1] = strip.(coalesce.(pr_df[!, 1], ""))
    for c in names(pr_df)[2:end]; rename!(pr_df, c => c*"_pre"); end

    gs_sub2 = copy(gs_sub)
    gs_sub2[!, 1] = strip.(coalesce.(gs_sub2[!, 1], ""))
    all_loci = get_dynamic_loci(gs_sub2)
    valid    = build_global_valid(gs_sub2, resolution)

    merged = innerjoin(gs_sub2, pr_df;
                       on=names(gs_sub2)[1]=>names(pr_df)[1], makeunique=true)
    isempty(merged) && return Dict{String,Float64}()

    corr = Dict(L => 0 for L in all_loci)
    tot  = Dict(L => 0 for L in all_loci)

    for row in eachrow(merged)
        for L in all_loci
            gs_raw = String[]
            for suf in ("",".1")
                col = L*suf
                col ∈ names(merged) && !ismissing(row[col]) &&
                    append!(gs_raw, strip.(split(string(row[col]),'/')))
            end
            isempty(gs_raw) && continue
            pr_raw = String[]
            for suf in ("_pre",".1_pre")
                col = L*suf
                col ∈ names(merged) && !ismissing(row[col]) &&
                    push!(pr_raw, strip(string(row[col])))
            end
            c, m, n, nc = score_pair(gs_raw, pr_raw, get(valid,L,Set{String}()), resolution)
            corr[L] += c; tot[L] += 2
        end
    end
    return Dict(L => (tot[L]>0 ? corr[L]/tot[L] : 0.0) for L in all_loci)
end

# Build per-locus ancestry accuracy
anc_locus_rows = []
if !isnothing(gs_eur)
    for t in TOOLS
        eur_l = compute_ancestry_locus_accuracy(gs_eur, t; resolution=2)
        yri_l = compute_ancestry_locus_accuracy(gs_yri, t; resolution=2)
        for L in ["A","B","C","DRB1","DQB1"]
            push!(anc_locus_rows, (Tool=t, Locus=L,
                                   Acc_Eur=get(eur_l,L,0.0),
                                   Acc_Yri=get(yri_l,L,0.0)))
        end
    end
end
df_anc_locus = DataFrame(anc_locus_rows)
println(df_anc_locus)

In [ ]:
# Fig 10 — side-by-side heatmaps: EUR and YRI per-locus accuracy
let
    isempty(df_anc_locus) && (println("No data"); return)

    LOCI_ORDER = ["A","B","C","DRB1","DQB1"]
    nl = length(LOCI_ORDER)
    nt = length(TOOLS)

    acc_eur = Dict((r.Tool, r.Locus) => r.Acc_Eur for r in eachrow(df_anc_locus))
    acc_yri = Dict((r.Tool, r.Locus) => r.Acc_Yri for r in eachrow(df_anc_locus))

    tool_ord = sort(TOOLS; by=t -> -mean(get(acc_eur,(t,L),0.0) for L in LOCI_ORDER))

    mat_e = [get(acc_eur,(t,L),0.0) for t in tool_ord, L in LOCI_ORDER]
    mat_y = [get(acc_yri,(t,L),0.0) for t in tool_ord, L in LOCI_ORDER]

    fig = Figure(size=(1000, 500))

    for (col, mat, pop) in [(1, mat_e, "European"), (2, mat_y, "Yoruba")]
        ax = Axis(fig[1,col];
                  title  = "Fig 10: 2-field Accuracy — $pop",
                  xticks = (1:nl, LOCI_ORDER),
                  yticks = (1:nt, tool_ord),
                  yticklabelsize=11)
        hm = heatmap!(ax, 1:nl, 1:nt, mat'; colormap=:YlGnBu, colorrange=(0,1))
        col==2 && Colorbar(fig[1,3], hm; label="Accuracy")
        for (i,L) in enumerate(LOCI_ORDER), (j,t) in enumerate(tool_ord)
            text!(ax, i, j; text=string(round(mat[j,i]; digits=2)),
                  align=(:center,:center), fontsize=8,
                  color=mat[j,i]>0.5 ? :black : :white)
        end
    end
    save("figures_julia/fig10_ancestry_locus.svg", fig)
    fig
end

# Figure 12: CPU and RAM Usage

In [ ]:
"""
Parse the segmented benchmark CSV (one block per sample).
Block format:
  Header: sample_id, tool1, tool2, ...
  Data:   iteration_num, val1, val2, ...
  avg row: skipped
  blank rows: skipped
Returns long-form DataFrame: sample, iteration, tool, raw_value.
"""
function parse_segmented_csv(path::String)::DataFrame
    isfile(path) || (println("Missing: $path"); return DataFrame())

    lines = readlines(path)
    records = []
    tools_row = String[]
    sample_id = ""

    for line in lines
        parts = split(line, ',')
        first_cell = strip(parts[1])

        isempty(first_cell) && continue                # blank row
        lowercase(first_cell) == "avg" && continue    # avg row

        if tryparse(Int, first_cell) === nothing
            # Header row: sample + tools
            sample_id = first_cell
            tools_row = [strip(p) for p in parts[2:end]]
        else
            # Data row: iteration + values
            iter = first_cell
            for (i, t) in enumerate(tools_row)
                isempty(t) && continue
                val = i+1 <= length(parts) ? strip(parts[i+1]) : ""
                isempty(val) && continue
                push!(records, (sample=sample_id, iteration=iter, tool=t, raw=val))
            end
        end
    end
    return DataFrame(records)
end

function hms_to_seconds(s::AbstractString)::Float64
    m = match(r"^(\d+):(\d+):(\d+\.?\d*)$", strip(s))
    m === nothing && return NaN
    return parse(Int,m[1])*3600 + parse(Int,m[2])*60 + parse(Float64,m[3])
end

function mem_to_gb(s::AbstractString)::Float64
    m = match(r"([\d.]+)\s*(MB|GB)"i, strip(s))
    m === nothing && return NaN
    v = parse(Float64, m[1])
    return uppercase(m[2]) == "GB" ? v : v/1024
end

# short name → match our TOOLS keys (the CSV uses display names)
const TOOL_NAME_MAP = Dict(
    "seq2HLA"        => "seq2hla",
    "rna2hla"        => "rna2hla",
    "arcasHLA"       => "arcas",
    "HLAforest"      => "hlaforest",
    "Optitype"       => "optitype",
    "Optitype "      => "optitype",
    "PHLAT"          => "phlat",
    "HLAMiner"       => "hlaminer",
    "HLA-HD"         => "hlahd",
    "HLAvbseq"       => "hlavbseq",
    "T1K"            => "T1K",
    "HISAT-genotype" => "hisat",
    "HLApers"        => "hlapers",
)

cpu_raw = parse_segmented_csv("../results/cpu_ram/runtime_data - ALL CPU DATA.csv")
ram_raw = parse_segmented_csv("../results/cpu_ram/runtime_data - ALL_RAM_DATA.csv")

if !isempty(cpu_raw)
    cpu_raw[!, :tool_key] = [get(TOOL_NAME_MAP, t, t) for t in cpu_raw.tool]
    cpu_raw[!, :cpu_s]    = hms_to_seconds.(cpu_raw.raw)
    cpu_long = dropmissing(filter(r -> !isnan(r.cpu_s), cpu_raw), :cpu_s)
    println("CPU rows: $(nrow(cpu_long))")
end
if !isempty(ram_raw)
    ram_raw[!, :tool_key] = [get(TOOL_NAME_MAP, t, t) for t in ram_raw.tool]
    ram_raw[!, :ram_gb]   = mem_to_gb.(ram_raw.raw)
    ram_long = dropmissing(filter(r -> !isnan(r.ram_gb), ram_raw), :ram_gb)
    println("RAM rows: $(nrow(ram_long))")
end

In [ ]:
# Fig 12 — CPU time boxplot + RAM boxplot side by side
let
    (!@isdefined(cpu_long) || isempty(cpu_long)) && (println("No CPU data"); return)
    (!@isdefined(ram_long) || isempty(ram_long)) && (println("No RAM data"); return)

    # median-sorted tool order for CPU
    cpu_med = Dict(t => median(filter(r->r.tool_key==t, cpu_long).cpu_s) for t in unique(cpu_long.tool_key))
    cpu_order = sort(collect(keys(cpu_med)); by=t->cpu_med[t])

    ram_med = Dict(t => median(filter(r->r.tool_key==t, ram_long).ram_gb) for t in unique(ram_long.tool_key))
    ram_order = sort(collect(keys(ram_med)); by=t->ram_med[t])

    fig = Figure(size=(1200, 500))

    # CPU
    ax1 = Axis(fig[1,1]; title="Fig 12a: CPU Runtime by Tool",
               ylabel="CPU time (s)", xticks=(1:length(cpu_order), cpu_order),
               xticklabelrotation=π/4)
    for (i, t) in enumerate(cpu_order)
        vals = filter(r->r.tool_key==t, cpu_long).cpu_s
        boxplot!(ax1, fill(i, length(vals)), vals;
                 color=tool_color(t), strokecolor=:black, strokewidth=0.5)
    end

    # RAM
    ax2 = Axis(fig[1,2]; title="Fig 12b: RAM Usage by Tool",
               ylabel="RAM (GB)", xticks=(1:length(ram_order), ram_order),
               xticklabelrotation=π/4)
    for (i, t) in enumerate(ram_order)
        vals = filter(r->r.tool_key==t, ram_long).ram_gb
        boxplot!(ax2, fill(i, length(vals)), vals;
                 color=tool_color(t), strokecolor=:black, strokewidth=0.5)
    end

    save("figures_julia/fig12_cpu_ram.svg", fig)
    fig
end

# Figure 13: Phase Ambiguities

In [ ]:
"""
Count phase-ambiguous predictions (allele strings containing '/') per tool.
"""
function compute_phase_ambiguities(;
        tools::Vector{String}=TOOLS,
        ds_nums=1:8,
        pr_dir::String=PR_DIR)::DataFrame

    CLASSICAL = ["A","B","C","DRB1","DQB1"]
    rows = []

    for t in tools
        total_calls = amb_calls = 0
        for ds in ds_nums
            pr_path = joinpath(pr_dir, "$(t)_d$(ds).csv")
            isfile(pr_path) || continue
            pr_df = CSV.read(pr_path, DataFrame; types=String)
            for L in CLASSICAL
                for suf in ("", ".1")
                    col = L * suf
                    col ∉ names(pr_df) && continue
                    col_data = collect(skipmissing(pr_df[!, col]))
                    col_data = filter(!isempty, strip.(col_data))
                    total_calls += length(col_data)
                    amb_calls   += count(x -> occursin('/', x), col_data)
                end
            end
        end
        pct = total_calls > 0 ? amb_calls / total_calls * 100 : 0.0
        push!(rows, (Tool=t, TotalCalls=total_calls, AmbiguousCalls=amb_calls, AmbPct=pct))
    end
    return sort!(DataFrame(rows), :AmbPct; rev=true)
end

df_phase = compute_phase_ambiguities()
println(df_phase)

In [ ]:
# Fig 13 — phase ambiguity rate bar chart
let
    n = nrow(df_phase)

    fig = Figure(size=(800, 500))
    ax  = Axis(fig[1,1];
               title  = "Fig 13: Phase Ambiguity Rate by Tool",
               ylabel = "Ambiguous Calls (%)",
               xticks = (1:n, df_phase.Tool),
               xticklabelrotation = π/4)
    barplot!(ax, 1:n, df_phase.AmbPct;
             color = [tool_color(t) for t in df_phase.Tool])
    for (i, r) in enumerate(eachrow(df_phase))
        r.AmbPct > 0 && text!(ax, i, r.AmbPct + 0.3;
                               text=string(round(r.AmbPct; digits=1))*"%",
                               align=(:center,:bottom), fontsize=9)
    end
    ylims!(ax, 0, nothing)
    save("figures_julia/fig13_phase_ambiguities.svg", fig)
    fig
end